## 1. 提取音频塔，将 `thinker` 前缀改为 `model`

最终保存的参数名称：

```text
thinker.audio_tower.conv2d1.weight
    ↓
model.audio_tower.conv2d1.weight
```

参数名称像地址。我们修改地址的第一级，保留音频塔的内部结构、参数值和 `forward`。

要让这个模型目录可以加载，需要三部分：

- 权重：保存训练得到的参数数值。
- config：描述音频塔的层数、宽度等结构。
- 模型类：建立结构，定义如何执行计算。

### 1.1 从已有分片读取音频塔参数

本例读取已下载的 `weights/model-00001-of-00015.safetensors`，其中包含完整音频塔参数。按名称筛选 `thinker.audio_tower.`，把参数放进 `source_state` 字典。


In [1]:
from pathlib import Path
from safetensors import safe_open

shard_path = Path("weights/model-00001-of-00015.safetensors")
export_dir = Path("weights/qwen3-omni-audio")
old_prefix = "thinker.audio_tower."
new_prefix = "model.audio_tower."

source_state = {}
with safe_open(filename = shard_path, framework = "pt", device = "cpu") as shard:
    for name in shard.keys():
        if name.startswith(old_prefix):
            source_state[name] = shard.get_tensor(name = name)

print("读取的音频参数张量数:", len(source_state))


读取的音频参数张量数: 525


### 1.2 取得音频塔的 config

原模型的 `config.json` 中，音频配置位于 `thinker_config.audio_config`。取出这一段，就有了建立音频塔需要的结构信息。


In [2]:
import json
from huggingface_hub import hf_hub_download

config_path = Path("weights/qwen3-omni-source/config.json")
if not config_path.is_file():
    config_path = Path(
        hf_hub_download(
            repo_id = "Qwen/Qwen3-Omni-30B-A3B-Instruct",
            filename = "config.json",
            local_dir = config_path.parent,
        )
    )

omni_config = json.loads(s = config_path.read_text())
audio_config_dict = omni_config["thinker_config"]["audio_config"]
print("音频塔层数:", audio_config_dict["encoder_layers"])


音频塔层数: 32


### 1.3 编写自定义配置类

新模型的配置中保存一份 `audio_config`，用它创建官方音频塔。`PretrainedConfig` 提供配置的保存和加载方法。

下面新增的字段各有一个用途：

- `model_type`：给这个自定义模型取一个名称。
- `audio_config`：保存原音频塔的配置。
- `sub_configs`：指定 `audio_config` 使用的配置类。

下面展示 `audio_tower_code/configuration_qwen3_omni_audio.py` 中的配置类。


```python
from transformers import PretrainedConfig
from transformers.models.qwen3_omni_moe.configuration_qwen3_omni_moe import (
    Qwen3OmniMoeAudioEncoderConfig,
)


class Qwen3OmniAudioTowerConfig(PretrainedConfig):
    model_type = "qwen3_omni_audio_tower"
    sub_configs = {"audio_config": Qwen3OmniMoeAudioEncoderConfig}

    def __init__(self, audio_config = None, **kwargs):
        super().__init__(**kwargs)
        if audio_config is None:
            audio_config = Qwen3OmniMoeAudioEncoderConfig()
        elif isinstance(audio_config, dict):
            audio_config = Qwen3OmniMoeAudioEncoderConfig.from_dict(
                config_dict = audio_config,
            )
        self.audio_config = audio_config
```


### 1.4 编写模型类，建立 `model.audio_tower`

`self.model` 是一个容器，`self.model.audio_tower` 中放官方音频编码器。这两个属性形成参数名称中的 `model.audio_tower.` 前缀。

```text
Qwen3OmniAudioTowerModel
└── model
    └── audio_tower ← 官方 Qwen3OmniMoeAudioEncoder
```

下面展示 `audio_tower_code/modeling_qwen3_omni_audio.py` 中的模型类，完成三件事：

1. 用 `config_class` 指定上一格的配置类。
2. 创建 `self.model.audio_tower`，并调用官方初始化方法 `post_init()`。
3. 在 `forward` 中将输入交给音频塔，返回它的输出。

`base_model_prefix` 标明父级属性名；`main_input_name` 标明主要输入名。`input_modalities` 标明音频类型；`_no_split_modules` 沿用音频层的设备分配设置。音频塔内部继续使用官方实现。


```python
from torch import nn
from transformers.models.qwen3_omni_moe.modeling_qwen3_omni_moe import (
    Qwen3OmniMoeAudioEncoder,
    Qwen3OmniMoePreTrainedModel,
)

from .configuration_qwen3_omni_audio import Qwen3OmniAudioTowerConfig


class Qwen3OmniAudioTowerModel(Qwen3OmniMoePreTrainedModel):
    config_class = Qwen3OmniAudioTowerConfig
    config: Qwen3OmniAudioTowerConfig
    base_model_prefix = "model"
    main_input_name = "input_features"
    input_modalities = ("audio",)
    _no_split_modules = ["Qwen3OmniMoeAudioEncoderLayer"]

    def __init__(self, config):
        super().__init__(config = config)
        self.model = nn.Module()
        self.model.audio_tower = Qwen3OmniMoeAudioEncoder(
            config = config.audio_config,
        )
        self.post_init()

    def forward(
        self,
        input_features = None,
        feature_lens = None,
        **kwargs,
    ):
        return self.model.audio_tower(
            input_features = input_features,
            feature_lens = feature_lens,
            **kwargs,
        )
```


### 1.5 替换参数前缀，装入新模型

`removeprefix` 删除指定的完整前缀，再接上 `model.audio_tower.`。参数张量直接沿用原始值。

`load_state_dict` 把参数装入对应的层。`strict = True` 要求名称和形状匹配；`assign = True` 保留读出的参数精度。原始参数使用 BF16，即 16 位浮点数。


In [3]:
from audio_tower_code.configuration_qwen3_omni_audio import Qwen3OmniAudioTowerConfig
from audio_tower_code.modeling_qwen3_omni_audio import Qwen3OmniAudioTowerModel

renamed_state = {
    new_prefix + name.removeprefix(old_prefix): tensor
    for name, tensor in source_state.items()
}
config = Qwen3OmniAudioTowerConfig(audio_config = audio_config_dict)
audio_model = Qwen3OmniAudioTowerModel(config = config)
loading_result = audio_model.load_state_dict(
    state_dict = renamed_state,
    strict = True,
    assign = True,
)
print(loading_result)

example = "thinker.audio_tower.conv2d1.weight"
print(example, "→", new_prefix + example.removeprefix(old_prefix))


<All keys matched successfully>
thinker.audio_tower.conv2d1.weight → model.audio_tower.conv2d1.weight


### 1.6 保存模型和音频 processor

音频预处理使用官方的 `WhisperFeatureExtractor`，负责把波形转换成音频塔需要的 Mel 特征。它的设置保存在 `preprocessor_config.json` 中，随模型一起保存。

下面先从原模型的预处理配置创建 `audio_processor`。随后，`register_for_auto_class` 登记模型类的自动加载入口，`save_pretrained` 保存模型、代码和音频预处理配置。`config.json` 中的 `auto_map` 记录自定义模型的“文件名.类名”。

```text
weights/qwen3-omni-audio/
├── config.json
├── model.safetensors
├── configuration_qwen3_omni_audio.py
├── modeling_qwen3_omni_audio.py
└── preprocessor_config.json
```



In [4]:
from transformers import WhisperFeatureExtractor

preprocessor_config_path = config_path.parent / "preprocessor_config.json"
if not preprocessor_config_path.is_file():
    hf_hub_download(
        repo_id = "Qwen/Qwen3-Omni-30B-A3B-Instruct",
        filename = "preprocessor_config.json",
        local_dir = config_path.parent,
    )

audio_processor = WhisperFeatureExtractor.from_pretrained(
    pretrained_model_name_or_path = config_path.parent,
)


In [5]:
Qwen3OmniAudioTowerConfig.register_for_auto_class(auto_class = "AutoConfig")
Qwen3OmniAudioTowerModel.register_for_auto_class(auto_class = "AutoModel")
audio_model.save_pretrained(save_directory = export_dir)
audio_processor.save_pretrained(save_directory = export_dir)

saved_config = json.loads(
    s = (export_dir / "config.json").read_text(),
)
print(json.dumps(obj = saved_config["auto_map"], indent = 2))


{
  "AutoConfig": "configuration_qwen3_omni_audio.Qwen3OmniAudioTowerConfig",
  "AutoModel": "modeling_qwen3_omni_audio.Qwen3OmniAudioTowerModel"
}


### 1.7 加载模型和音频 processor

`AutoModel` 根据 `auto_map` 找到模型目录中的自定义类。`trust_remote_code = True` 允许执行目录里的 `.py`。`AutoFeatureExtractor` 读取 `preprocessor_config.json`，加载音频 processor。

下面使用 FP32，也就是 32 位浮点数，在 CPU 上运行示例。`eager` 选择普通的注意力计算实现，`eval()` 切换到推理模式。加载后的音频塔位于 `reloaded_model.model.audio_tower`。


In [6]:
import torch
from transformers import AutoModel, AutoFeatureExtractor

audio_processor = AutoFeatureExtractor.from_pretrained(
    pretrained_model_name_or_path = export_dir,
)

reloaded_model = AutoModel.from_pretrained(
    pretrained_model_name_or_path = export_dir,
    trust_remote_code = True,
    dtype = torch.float32,
    attn_implementation = "eager",
)
reloaded_model.eval()
reloaded_model


Qwen3OmniAudioTowerModel(
  (model): Module(
    (audio_tower): Qwen3OmniMoeAudioEncoder(
      (positional_embedding): SinusoidsPositionEmbedding()
      (layers): ModuleList(
        (0-31): 32 x Qwen3OmniMoeAudioEncoderLayer(
          (self_attn): Qwen3OmniMoeAudioAttention(
            (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
            (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1280, out_features=5120, bias=True)
          (fc2): Linear(in_features=5120, out_features=1280, bias=True)
          (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True, bias=True)
        )
    

### 1.8 最小 forward 调用

官方音频塔接收整理好的音频特征和每条音频的帧数。下面用全零特征演示调用：一条音频、100 帧，每帧 128 个数。

`input_features` 的形状为 `[128, 100]`，`feature_lens` 为 `[100]`。返回值中的 `last_hidden_state` 保存音频编码特征。


In [7]:
input_features = torch.zeros(size = (128, 100))
feature_lens = torch.tensor(data = [100])

with torch.inference_mode():
    outputs = reloaded_model(
        input_features = input_features,
        feature_lens = feature_lens,
    )

print(outputs.last_hidden_state.shape)


torch.Size([13, 2048])


输出形状为 `[13, 2048]`：音频塔沿时间方向压缩后得到 13 个位置，每个位置有 2048 个特征值。

参考：[官方模型配置](https://huggingface.co/Qwen/Qwen3-Omni-30B-A3B-Instruct/blob/main/config.json)、[自定义模型的保存与加载](https://huggingface.co/docs/transformers/custom_models)。


## 2. 准备语言模型骨干

本节以 Qwen3.5-0.8B 为例，先加载本地模型，再读取文本模型的特征维度，用来确定 projector 的输出维度。


In [8]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    pretrained_model_name_or_path = "weights/Qwen3.5-0.8B",
)


### 2.1 确定 LLM 的特征维度

LLM 用一组数字表示每个 token，这组数字的个数就是特征维度 `hidden_size`。Qwen3.5 的文本模型配置位于 `model.config.text_config`，从中读取 `hidden_size` 就能得到这个维度。

Projector 输出的每个音频位置也需要用同样数量的数字表示，才能接入 LLM，因此它的输出维度取这个 `hidden_size`。


In [9]:
llm_hidden_size = model.config.text_config.hidden_size
projector_output_dim = llm_hidden_size

print("LLM 特征维度:", llm_hidden_size)
print("Projector 输出维度:", projector_output_dim)


LLM 特征维度: 1024
Projector 输出维度: 1024


## 3. 准备音频 projector

音频塔每个位置输出 2048 个特征值，LLM 每个位置使用 1024 个特征值。Projector 将这组音频特征映射到 LLM 的维度，保留原来的位置数量。本节使用两层 MLP：`2048 → 1024 → 1024`，中间经过 SiLU。

### 3.1 目录与命名

源码直接放在项目根目录下的 `qwen3.5-0.8b-mllm/`。采用 Hugging Face 的文件命名方式，后续在同一目录中补齐完整模型：

```text
qwen3.5-0.8b-mllm/
├── modeling_qwen3_5_mllm.py       本节：projector；后续加入完整模型类
├── configuration_qwen3_5_mllm.py 后续：完整模型配置类
├── processing_qwen3_5_mllm.py    后续：完整模型的输入处理
├── config.json                 后续：配置和自动加载入口
└── model.safetensors           后续：完整模型权重
```

完整模型的类名采用 `Qwen3_5MLLMForConditionalGeneration`，配置类采用 `Qwen3_5MLLMConfig`，`model_type` 采用 `qwen3_5_mllm`。将来合并时，模块放在以下位置：

```text
Qwen3_5MLLMForConditionalGeneration
├── model
│   ├── audio_tower       ← 已独立导出的音频塔
│   ├── audio_projector   ← 本节的 projector
│   ├── visual          ← Qwen3.5 原有的图像、视频视觉塔
│   └── language_model
└── lm_head
```

Projector 的两层命名为 `linear_1` 和 `linear_2`。合入完整模型后，参数名称就是 `model.audio_projector.linear_1.*` 和 `model.audio_projector.linear_2.*`。音频塔沿用现有的 `model.audio_tower.*`。

Hugging Face 与 vLLM 的加载映射见 4.10 节。


### 3.2 读取输入和输出维度

输入维度取音频塔配置中的 `output_dim`；输出维度沿用第二节从 LLM 的 `hidden_size` 得到的 `projector_output_dim`。


In [10]:
projector_input_dim = reloaded_model.config.audio_config.output_dim

print("Projector 输入维度:", projector_input_dim)
print("Projector 输出维度:", projector_output_dim)


Projector 输入维度: 2048
Projector 输出维度: 1024


### 3.3 定义两层 MLP

下面展示 `qwen3.5-0.8b-mllm/modeling_qwen3_5_mllm.py` 中的 `Qwen3_5MLLMAudioProjector`。第一层 `linear_1` 将 2048 维映射到 1024 维；SiLU 加入非线性变换，让两层可以学习更灵活的关系；第二层 `linear_2` 继续处理这 1024 个特征值。

这个类继承 `nn.Module`，以后可以直接挂到完整模型的 `model.audio_projector` 属性上。`forward` 接收音频塔的输出，返回映射后的特征。


```python
from torch import nn


class Qwen3_5MLLMAudioProjector(nn.Module):
    """将音频塔输出映射到语言模型的特征维度。"""

    def __init__(self, audio_output_dim, text_hidden_size):
        super().__init__()
        self.linear_1 = nn.Linear(
            in_features = audio_output_dim,
            out_features = text_hidden_size,
            bias = True,
        )
        self.silu = nn.SiLU()
        self.linear_2 = nn.Linear(
            in_features = text_hidden_size,
            out_features = text_hidden_size,
            bias = True,
        )

    def forward(self, audio_features):
        hidden_states = self.linear_1(input = audio_features)
        hidden_states = self.silu(input = hidden_states)
        return self.linear_2(input = hidden_states)
```


### 3.4 创建 projector

用 Transformers 的动态模块加载方法，从模型目录读取这个类，再传入前两节确定的维度。`class_reference` 写成“文件名.类名”。


In [11]:
from transformers.dynamic_module_utils import get_class_from_dynamic_module

Qwen3_5MLLMAudioProjector = get_class_from_dynamic_module(
    class_reference = "modeling_qwen3_5_mllm.Qwen3_5MLLMAudioProjector",
    pretrained_model_name_or_path = "qwen3.5-0.8b-mllm",
)
projector = Qwen3_5MLLMAudioProjector(
    audio_output_dim = projector_input_dim,
    text_hidden_size = projector_output_dim,
)
projector


Qwen3_5MLLMAudioProjector(
  (linear_1): Linear(in_features=2048, out_features=1024, bias=True)
  (silu): SiLU()
  (linear_2): Linear(in_features=1024, out_features=1024, bias=True)
)

### 3.5 将音频塔输出送入 projector

第一节的 `outputs.last_hidden_state` 是音频塔输出，形状为 `[13, 2048]`。送入 projector 后，形状变为 `[13, 1024]`：13 个音频位置保留，每个位置的特征宽度变为 LLM 的 1024 维。


In [12]:
audio_features = outputs.last_hidden_state

with torch.inference_mode():
    projected_audio_features = projector(audio_features = audio_features)

print("音频塔输出:", audio_features.shape)
print("Projector 输出:", projected_audio_features.shape)


音频塔输出: torch.Size([13, 2048])
Projector 输出: torch.Size([13, 1024])


Projector 的参数是新初始化的，后续需要训练来学习音频特征与文本特征之间的对应关系。

参考：[Hugging Face 自定义模型](https://huggingface.co/docs/transformers/custom_models)、[vLLM 多模态模型接入](https://docs.vllm.ai/en/latest/contributing/model/multimodal/)。


## 4. 将音频、视觉组件放入模型目录

本节在 Qwen3.5 原有文本、图像和视频处理的基础上加入音频组件，统一使用 `qwen3.5-0.8b-mllm`。MLLM 是多模态大语言模型的缩写；本项目的目标是文本、图像、视频和音频输入，文本输出。接入 LLM 的完整 `forward`、`generate` 和合并权重留到下一步。

### 4.1 文件各自负责什么

```text
qwen3.5-0.8b-mllm/
├── configuration_qwen3_omni_audio.py  第一节的独立音频塔配置类
├── modeling_qwen3_omni_audio.py       第一节的独立音频塔模型类
├── modeling_qwen3_5_mllm.py           第三节的 SiLU projector
├── configuration_qwen3_5_mllm.py      文本、视觉、音频的总配置类
├── processing_qwen3_5_mllm.py         文本、图像、视频、音频输入处理类
├── config.json                       总配置与 AutoConfig 加载入口
├── processor_config.json             图像、视频、音频预处理设置和加载入口
├── tokenizer.json                    Qwen3.5 的词表和分词规则
├── tokenizer_config.json             tokenizer 的设置
└── chat_template.jinja               Qwen3.5 原有的文本、图像、视频对话模板
```

前两份 `.py` 沿用第一节 1.3、1.4 展示的代码。音频塔仍位于包装类的 `model.audio_tower`，内部结构和 `forward` 保持原样。Projector 沿用第三节的 `Linear → SiLU → Linear`。

视觉塔复用 Transformers 的 `Qwen3_5VisionModel`。第二节加载的 `model.visual` 已包含它的预训练权重，图像和视频共用这座视觉塔。完整模型中继续保留 `model.visual`，其原有的视觉特征合并层也一起保留。

音频权重从 `weights/qwen3-omni-audio/` 读取，视觉和语言模型权重从 `weights/Qwen3.5-0.8B/` 读取。新目录保存组件源码和配置，完整模型权重在下一步组装后保存。


### 4.2 为总配置加入音频塔配置

这里先说明组件准备阶段的配置。第五节展示加入音频标记编号后的配置类。

配置可以看作搭建模型的说明书：`audio_config` 说明怎样搭音频塔，`text_config` 说明怎样搭语言模型，`vision_config` 说明怎样搭视觉塔。下面的 `configuration_qwen3_5_mllm.py` 继承 Qwen3.5 的配置类，保留原有文本、视觉设置，并加入 `audio_config`。

`model_type = "qwen3_5_mllm"` 是这份总配置的名称。`sub_configs` 声明各份子配置对应的类；从 JSON 读到字典时，`__init__` 会把音频配置转换为官方配置对象。Projector 的输入和输出维度分别读取 `audio_config.output_dim` 与 `text_config.hidden_size`，两层结构和 SiLU 继续由第三节的源码定义。


```python
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5Config
from transformers.models.qwen3_omni_moe.configuration_qwen3_omni_moe import (
    Qwen3OmniMoeAudioEncoderConfig,
)


class Qwen3_5MLLMConfig(Qwen3_5Config):
    """在 Qwen3.5 配置中加入音频塔配置。"""

    model_type = "qwen3_5_mllm"
    sub_configs = {
        **Qwen3_5Config.sub_configs,
        "audio_config": Qwen3OmniMoeAudioEncoderConfig,
    }

    def __init__(self, audio_config = None, **kwargs):
        super().__init__(**kwargs)
        if audio_config is None:
            audio_config = Qwen3OmniMoeAudioEncoderConfig()
        elif isinstance(audio_config, dict):
            audio_config = Qwen3OmniMoeAudioEncoderConfig.from_dict(
                config_dict = audio_config,
            )
        self.audio_config = audio_config
```


### 4.3 生成 config.json

沿用第三节的动态加载方法读取新类，再从两个权重目录读取 backbone 和音频塔的配置。将 backbone 配置转成字典，去掉原模型的类型和类名，再加入音频配置，得到 `mllm_config`。

`register_for_auto_class` 和 `save_pretrained` 会把自定义配置类的加载入口写入 `auto_map`。本节登记 `AutoConfig`，完整模型类的入口在下一步实现后登记。


In [13]:
from transformers import AutoConfig, AutoProcessor

mllm_dir = Path("qwen3.5-0.8b-mllm")

Qwen3_5MLLMConfig = get_class_from_dynamic_module(
    class_reference = "configuration_qwen3_5_mllm.Qwen3_5MLLMConfig",
    pretrained_model_name_or_path = mllm_dir,
)

backbone_config = AutoConfig.from_pretrained(
    pretrained_model_name_or_path = "weights/Qwen3.5-0.8B",
)
audio_tower_config = AutoConfig.from_pretrained(
    pretrained_model_name_or_path = export_dir,
    trust_remote_code = True,
)
backbone_config_dict = backbone_config.to_dict()
backbone_config_dict.pop("model_type", None)
backbone_config_dict.pop("architectures", None)

mllm_config = Qwen3_5MLLMConfig(
    audio_config = audio_tower_config.audio_config.to_dict(),
    **backbone_config_dict,
)
Qwen3_5MLLMConfig.register_for_auto_class(auto_class = "AutoConfig")
mllm_config.save_pretrained(save_directory = mllm_dir)

print("音频塔输出维度:", mllm_config.audio_config.output_dim)
print("LLM 特征维度:", mllm_config.text_config.hidden_size)


音频塔输出维度: 2048
LLM 特征维度: 1024


### 4.4 组合文本、图像、视频和音频 processor

下面展示组件准备阶段的实现。第五节在此基础上加入音频标记展开和音视频时间对齐，并展示当前文件的完整实现。

`processing_qwen3_5_mllm.py` 把四种输入放到一个入口。`tokenizer` 将文本切成称为 token 的小片段，再转换为编号；`image_processor` 和 `video_processor` 将图像、视频整理成视觉塔接收的像素块；`feature_extractor` 将波形转换成 Mel 特征，也就是按时间排列的声音频率信息。`ProcessorMixin` 提供一起保存、重新加载这些设置的方法。

`vision_processor` 使用原有的 `tokenizer`、`image_processor`、`video_processor` 构建官方 `Qwen3VLProcessor`。文本、图像和视频交给它处理，沿用官方逻辑：按图像需要的特征数量重复图像标记，为视频帧加入时间信息。调用时分别使用 `text`、`images`、`videos` 和 `audio`，可以组合传入。

音频继续使用 `WhisperFeatureExtractor`：`padding = "longest"` 将同批音频补齐到最长长度，`truncation = False` 保留完整音频。`sampling_rate` 指明波形的采样率，这里的音频塔使用 16000 Hz，即每秒 16000 个采样点。`feature_attention_mask` 用 1 标记有效帧、用 0 标记补齐帧，文本的 `attention_mask` 保持原名。

`return_tensors = "pt"` 让输出使用 PyTorch 张量，额外的 `kwargs` 交给官方视觉 processor，供文本分词、图像和视频处理使用。下一步接入 LLM 时，再安排音频特征在输入序列中的位置。


```python
from transformers import ProcessorMixin, Qwen3VLProcessor
from transformers.feature_extraction_utils import BatchFeature


class Qwen3_5MLLMProcessor(ProcessorMixin):
    """保留 Qwen3.5 的文本、图像和视频处理，并加入音频处理。"""

    def __init__(
        self,
        image_processor,
        tokenizer,
        video_processor,
        feature_extractor,
        chat_template = None,
    ):
        super().__init__(
            image_processor = image_processor,
            tokenizer = tokenizer,
            video_processor = video_processor,
            feature_extractor = feature_extractor,
            chat_template = chat_template,
        )

    @property
    def vision_processor(self):
        return Qwen3VLProcessor(
            image_processor = self.image_processor,
            tokenizer = self.tokenizer,
            video_processor = self.video_processor,
            chat_template = self.chat_template,
        )

    def __call__(
        self,
        text = None,
        images = None,
        videos = None,
        audio = None,
        sampling_rate = None,
        return_tensors = "pt",
        **kwargs,
    ):
        if text is None and images is None and videos is None and audio is None:
            raise ValueError("请提供 text、images、videos 或 audio。")

        inputs = BatchFeature(data = {})
        if text is not None or images is not None or videos is not None:
            inputs = self.vision_processor(
                text = text,
                images = images,
                videos = videos,
                return_tensors = return_tensors,
                **kwargs,
            )

        if audio is not None:
            if sampling_rate is None:
                raise ValueError("传入 audio 时，请用 sampling_rate 指明波形的采样率。")
            audio_inputs = self.feature_extractor(
                raw_speech = audio,
                sampling_rate = sampling_rate,
                padding = "longest",
                truncation = False,
                return_attention_mask = True,
                return_tensors = return_tensors,
            )
            inputs["input_features"] = audio_inputs["input_features"]
            inputs["feature_attention_mask"] = audio_inputs["attention_mask"]

        return inputs

    @property
    def model_input_names(self):
        return self.vision_processor.model_input_names + [
            "input_features",
            "feature_attention_mask",
        ]
```


### 4.5 保存完整的 processor

从 Qwen3.5 权重目录加载官方 processor，取出它的 tokenizer、图像处理器、视频处理器和对话模板，再加入第一节的音频特征提取器。

保存后，`processor_config.json` 包含 `image_processor`、`video_processor`、`feature_extractor` 三份预处理配置，tokenizer 的词表与设置另行保存。`AutoProcessor` 的入口指向 `Qwen3_5MLLMProcessor`，重新加载时就能还原这些组件。


In [14]:
Qwen3_5MLLMProcessor = get_class_from_dynamic_module(
    class_reference = "processing_qwen3_5_mllm.Qwen3_5MLLMProcessor",
    pretrained_model_name_or_path = mllm_dir,
)
backbone_processor = AutoProcessor.from_pretrained(
    pretrained_model_name_or_path = "weights/Qwen3.5-0.8B",
)
mllm_processor = Qwen3_5MLLMProcessor(
    image_processor = backbone_processor.image_processor,
    tokenizer = backbone_processor.tokenizer,
    video_processor = backbone_processor.video_processor,
    feature_extractor = audio_processor,
    chat_template = backbone_processor.chat_template,
)
Qwen3_5MLLMProcessor.register_for_auto_class(auto_class = "AutoProcessor")
mllm_processor.save_pretrained(save_directory = mllm_dir)


['qwen3.5-0.8b-mllm/processor_config.json']

### 4.6 从新目录读取配置、processor 和音频塔源码

`AutoConfig`、`AutoProcessor` 读取刚保存的加载入口。`trust_remote_code = True` 允许加载目录里的自定义 `.py`。音频塔的类也从新目录读取，参数来自第一节导出的权重目录。


In [15]:
mllm_config = AutoConfig.from_pretrained(
    pretrained_model_name_or_path = mllm_dir,
    trust_remote_code = True,
)
mllm_processor = AutoProcessor.from_pretrained(
    pretrained_model_name_or_path = mllm_dir,
    trust_remote_code = True,
)

Qwen3OmniAudioTowerModel = get_class_from_dynamic_module(
    class_reference = "modeling_qwen3_omni_audio.Qwen3OmniAudioTowerModel",
    pretrained_model_name_or_path = mllm_dir,
)
audio_component = Qwen3OmniAudioTowerModel.from_pretrained(
    pretrained_model_name_or_path = export_dir,
    dtype = torch.float32,
    attn_implementation = "eager",
)
audio_component.eval()

mllm_projector = Qwen3_5MLLMAudioProjector(
    audio_output_dim = mllm_config.audio_config.output_dim,
    text_hidden_size = mllm_config.text_config.hidden_size,
)
mllm_projector.eval()


Qwen3_5MLLMAudioProjector(
  (linear_1): Linear(in_features=2048, out_features=1024, bias=True)
  (silu): SiLU()
  (linear_2): Linear(in_features=1024, out_features=1024, bias=True)
)

### 4.7 把波形转换成音频特征

用一秒静音演示输入：16000 个全零采样点，采样率为 16000 Hz。Processor 同时处理文本和这段音频，得到文本 token、Mel 特征和有效帧标记。

这段音频的 `input_features` 形状为 `[1, 128, 100]`，分别表示 1 条音频、128 个频率特征、100 帧。`feature_attention_mask` 的形状为 `[1, 100]`，对应这一条音频的 100 帧。


In [16]:
import numpy as np

waveform = np.zeros(shape = 16000, dtype = np.float32)
mllm_inputs = mllm_processor(
    text = "请把这段语音转写成文字。",
    audio = waveform,
    sampling_rate = 16000,
    return_tensors = "pt",
)

print("文本 token 形状:", mllm_inputs["input_ids"].shape)
print("Mel 特征形状:", mllm_inputs["input_features"].shape)
print("音频帧标记形状:", mllm_inputs["feature_attention_mask"].shape)


文本 token 形状: torch.Size([1, 22])
Mel 特征形状: torch.Size([1, 128, 100])
音频帧标记形状: torch.Size([1, 100])


### 4.8 依次送入音频塔和 projector

音频塔的原始 `forward` 接收 `[128, 总有效帧数]`。先把 Mel 特征从 `[音频条数, 128, 帧数]` 调整成 `[音频条数, 帧数, 128]`，这样就能用帧标记取出有效帧。取出后再把 128 这一维移到前面，得到音频塔需要的形状。`feature_lens` 记录每条音频的有效帧数；本例输入形状为 `[128, 100]`，`feature_lens` 的值为 `[100]`。

**100 帧为什么得到 13 个音频位置？** 官方音频塔依次经过 `conv2d1`、`conv2d2`、`conv2d3` 三层卷积。每层的卷积核大小为 3，步长为 2，边界补零 `padding = 1`。沿时间方向，每层输出长度都等于输入长度除以 2 后向上取整：

```text
输入       第一层卷积    第二层卷积    第三层卷积
100 帧  →     50      →     25      →     13
```

对本例这一个 100 帧块，可以用 `100 ÷ 8 = 12.5`，再向上取整得到 13 来理解数量。实际计算时，padding 加在每层卷积输入的边界，每层直接产生整数个输出位置；第 13 个位置也是根据末尾帧和边界补零计算出来的有效音频特征。

当前配置的 `n_window = 50`，音频塔按 `2 × n_window = 100` 帧分块进行卷积，再拼接各块的有效输出。因此，更长音频要按块计算长度，例如 200 帧分成两个 100 帧块，输出 `13 + 13 = 26` 个位置。[官方音频塔源码](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_omni_moe/modeling_qwen3_omni_moe.py)

为对齐块长度而补齐的无效输出位置，会在进入后续编码层前筛掉。后续编码层保留本例的这 13 个有效位置，每个位置最终输出 2048 个特征值，因此得到 `[13, 2048]`。Projector 将每个位置映射到 LLM 的 1024 维，得到 `[13, 1024]`。Projector 的参数仍是新初始化的，后续需要训练。


In [17]:
feature_mask = mllm_inputs["feature_attention_mask"].bool()
feature_lens = feature_mask.sum(dim = -1)
mel_features = mllm_inputs["input_features"].transpose(dim0 = 1, dim1 = 2)
packed_features = mel_features[feature_mask].transpose(dim0 = 0, dim1 = 1)
packed_features = packed_features.contiguous()

with torch.inference_mode():
    audio_outputs = audio_component(
        input_features = packed_features,
        feature_lens = feature_lens,
    )
    mllm_audio_features = mllm_projector(
        audio_features = audio_outputs.last_hidden_state,
    )

print("每条音频的有效帧数:", feature_lens.tolist())
print("音频塔输入:", packed_features.shape)
print("音频塔输出:", audio_outputs.last_hidden_state.shape)
print("Projector 输出:", mllm_audio_features.shape)


每条音频的有效帧数: [100]
音频塔输入: torch.Size([128, 100])
音频塔输出: torch.Size([13, 2048])
Projector 输出: torch.Size([13, 1024])


### 4.9 图像和视频送入原有视觉塔

第二节的 `model` 是完整的 Qwen3.5 backbone：`model.language_model` 负责语言，`model.visual` 负责图像和视频。这里直接使用 `model.visual`，保留它的结构、参数和原有 `forward`。

先用一张 256 × 256 的黑色图片演示。文本中的 `<|image_pad|>` 标记图片的位置，官方 processor 会按图片需要的特征数量展开这个标记。`pixel_values` 保存处理后的像素块；`image_grid_thw` 记录这些块在时间、高度、宽度上的排列，视觉塔据此计算位置。

视觉塔返回的 `pooler_output` 已经过原有的视觉特征合并层，宽度为 LLM 的 1024 维。本例得到 64 个视觉位置，即 `[64, 1024]`。


In [18]:
sample_image = np.zeros(shape = (256, 256, 3), dtype = np.uint8)
image_inputs = mllm_processor(
    text = "<|vision_start|><|image_pad|><|vision_end|>请描述图片。",
    images = sample_image,
    return_tensors = "pt",
)
vision_tower = model.visual
vision_tower.eval()

with torch.inference_mode():
    image_outputs = vision_tower(
        hidden_states = image_inputs["pixel_values"].to(dtype = vision_tower.dtype),
        grid_thw = image_inputs["image_grid_thw"],
        return_dict = True,
    )

print("图像网格:", image_inputs["image_grid_thw"].tolist())
print("图像特征:", image_outputs.pooler_output.shape)


图像网格: [[1, 16, 16]]
图像特征: torch.Size([64, 1024])


再把同一张黑色图片重复四次，组成四帧视频，帧率设为每秒 2 帧。`video_metadata` 提供帧率和帧编号，官方 processor 用它们确定每帧的时间；`do_sample_frames = False` 使用这里提供的全部四帧。

视频处理结果使用 `pixel_values_videos` 和 `video_grid_thw`，随后送入同一座 `vision_tower`。本例四帧按每两帧一组处理，得到 `[128, 1024]` 的特征。


In [19]:
sample_video = np.stack(arrays = [sample_image] * 4, axis = 0)
video_metadata = [{
    "fps": 2.0,
    "frames_indices": [0, 1, 2, 3],
    "total_num_frames": 4,
}]
video_inputs = mllm_processor(
    text = "<|vision_start|><|video_pad|><|vision_end|>请描述视频。",
    videos = sample_video,
    video_metadata = video_metadata,
    do_sample_frames = False,
    return_tensors = "pt",
)
video_pixels = video_inputs["pixel_values_videos"].to(dtype = vision_tower.dtype)

with torch.inference_mode():
    video_outputs = vision_tower(
        hidden_states = video_pixels,
        grid_thw = video_inputs["video_grid_thw"],
        return_dict = True,
    )

print("视频网格:", video_inputs["video_grid_thw"].tolist())
print("视频特征:", video_outputs.pooler_output.shape)


视频网格: [[2, 16, 16]]
视频特征: torch.Size([128, 1024])


参考：[Hugging Face 自定义配置与自动加载](https://huggingface.co/docs/transformers/custom_models)、[Processor 的保存与加载](https://huggingface.co/docs/transformers/main_classes/processors)。

视觉部分参考：[Qwen3.5 模型与视觉接口](https://huggingface.co/docs/transformers/model_doc/qwen3_5)。


### 4.10 Hugging Face、vLLM 的命名与自动加载

自动加载需要一份“类的地址簿”：配置文件写明使用哪个 Python 文件里的哪个类，框架据此找到代码。本项目沿用 Hugging Face 常见的文件与类命名方式：

| 用途 | Python 文件 | 类名 |
| --- | --- | --- |
| 总配置 | `configuration_qwen3_5_mllm.py` | `Qwen3_5MLLMConfig` |
| 输入处理 | `processing_qwen3_5_mllm.py` | `Qwen3_5MLLMProcessor` |
| 音频 projector | `modeling_qwen3_5_mllm.py` | `Qwen3_5MLLMAudioProjector` |
| 完整模型 | `modeling_qwen3_5_mllm.py` | `Qwen3_5MLLMForConditionalGeneration` |

目录 `qwen3.5-0.8b-mllm/` 用作加载路径；Python 文件名使用下划线，类名使用大小写分词。`configuration_`、`modeling_`、`processing_` 是便于识别职责的命名惯例，自动加载时需要让配置里的文件名、类名与实际代码逐字匹配，包括大小写。

**Hugging Face：通过对象修改配置并保存**

4.3 和 4.5 已经通过 Transformers 对象构建配置与 processor，再调用 `save_pretrained` 保存。下面的 JSON 是库生成文件的字段摘录，用来说明自动加载如何找到源码。

这些对象建立后，可以只给指定属性赋值。例如，下面仅替换音频子配置和音频特征提取器，原有文本、视觉子配置，以及 tokenizer、图像处理器、视频处理器继续沿用：

```python
mllm_config.audio_config = audio_tower_config.audio_config
mllm_processor.feature_extractor = audio_processor

mllm_config.save_pretrained(save_directory = mllm_dir)
mllm_processor.save_pretrained(save_directory = mllm_dir)
```

`register_for_auto_class` 负责登记自定义类，`save_pretrained` 负责写出配置和加载映射。保存时，框架会维护 `auto_map`、`processor_class` 等加载信息，JSON 的格式或默认字段也可能调整；其他组件的参数设置沿用原对象。

**auto_map 如何定位自定义类**

`model_type = "qwen3_5_mllm"` 标识这类模型的配置，应与配置类中的 `model_type` 一致。下面是 `config.json` 中配置类的加载字段：

```json
{
  "model_type": "qwen3_5_mllm",
  "auto_map": {
    "AutoConfig": "configuration_qwen3_5_mllm.Qwen3_5MLLMConfig"
  }
}
```

映射值采用 `Python文件名.类名`，文件名省略 `.py`。调用 `AutoConfig.from_pretrained` 并设置 `trust_remote_code = True` 后，框架读取 `auto_map.AutoConfig`，从该目录的 `configuration_qwen3_5_mllm.py` 加载 `Qwen3_5MLLMConfig`。4.3 的 `register_for_auto_class` 配合 `save_pretrained` 已写入这个映射。[Hugging Face 自定义模型说明](https://huggingface.co/docs/transformers/custom_models)

`processor_config.json` 中的 processor 加载字段如下：

```json
{
  "processor_class": "Qwen3_5MLLMProcessor",
  "auto_map": {
    "AutoProcessor": "processing_qwen3_5_mllm.Qwen3_5MLLMProcessor"
  }
}
```

`AutoProcessor` 按这个地址加载自定义 processor，再读取其图像、视频、音频设置及 tokenizer 文件，恢复各个输入处理组件。完整模型类实现后，还会登记对应的 `AutoModel`／任务专用 AutoModel 加载入口，并通过模型类的 `config_class` 关联 `Qwen3_5MLLMConfig`。

模块内部的属性名同时决定权重名称：`self.model.audio_tower` 对应 `model.audio_tower.*`，`self.model.audio_projector` 对应 `model.audio_projector.*`，视觉塔沿用 `model.visual.*`。加载权重时，这些名称需要与实际模块层级一致。

**vLLM：通过架构名选择模型实现**

采用 vLLM 原生模型接入方式时，`config.json` 的 `architectures` 记录完整模型的架构名。完整模型调用 `save_pretrained` 时，会将模型类名记录到 `config.architectures`，字段如下：

```json
{
  "architectures": ["Qwen3_5MLLMForConditionalGeneration"]
}
```

vLLM 用这个名称查找 `ModelRegistry` 中同名的注册项，再加载对应的 vLLM 模型实现。外部实现可以通过插件调用 `ModelRegistry.register_model` 登记；延迟加载地址采用 `Python包.模块:类名`，这里用冒号分隔模块和类。注册名必须与 `architectures` 中的名称匹配。[vLLM 模型注册说明](https://docs.vllm.ai/en/latest/contributing/model/registration/)

多模态接入还需要实现 `SupportsMultiModal`，并用 `MULTIMODAL_REGISTRY` 登记图像、视频、音频的处理规则，让 vLLM 知道如何将这些输入接到语言模型中。[vLLM 多模态接入说明](https://docs.vllm.ai/en/latest/contributing/model/multimodal/)

vLLM 也提供 Transformers 后端。该路径要求自定义基础模型满足后端接口，并在 `config.json` 中提供 `auto_map.AutoModel`；具体兼容性需要在完整模型实现后确认。目前本目录已提供 `AutoConfig` 和 `AutoProcessor` 两个自动加载入口。[vLLM Transformers 后端说明](https://docs.vllm.ai/en/latest/models/supported_models/#transformers-backend)


## 5. 融合音频塔、视觉塔与语言模型

本节把前四节的组件装入一个模型。文本先转成词向量，音频经过音频塔和 projector，图像、视频经过 Qwen3.5 原有视觉塔；这些特征放进同一条输入序列，再交给语言模型。`lm_head` 是文本输出层，将语言模型的输出转换为词表中各个 token 的分数。

```text
Qwen3_5MLLMForConditionalGeneration
├── model：Qwen3_5MLLMModel
│   ├── visual           Qwen3.5 原有视觉塔及视觉特征合并层
│   ├── language_model   Qwen3.5 原有语言模型
│   ├── audio_tower      Qwen3-Omni 原有音频塔
│   └── audio_projector  第三节的 Linear → SiLU → Linear
└── lm_head              Qwen3.5 原有文本输出层
```

源码继续放在 `qwen3.5-0.8b-mllm/`，配置通过 Transformers 对象更新。本节同时处理音频的位置编码，以及同一段视频中画面与声音的时间对齐。


### 5.1 给音频特征安排输入位置

Tokenizer 已有 `<|audio_start|>`、`<|audio_pad|>`、`<|audio_end|>`。前后两个标记圈出音频范围，中间的 `<|audio_pad|>` 表示等待填入音频特征的位置。Processor 从 `feature_attention_mask` 统计有效 Mel 帧数，再调用官方 `_get_feat_extract_output_lengths` 计算音频塔的输出长度，并展开相同数量的占位标记。源码中的 `tokenizer.audio_token` 就是 `<|audio_pad|>`，已注册为特殊 token，编号为 `248076`。模型通过配置中的 `audio_token_id` 找到这些位置，再用 projector 输出的向量填入，词表大小保持原样。

音频处理沿用官方的分块、卷积 padding、有效位置筛选和 `forward`。例如，100 个有效 Mel 帧产生 13 个特征，200 帧分成两块后产生 26 个特征。纯音频按这些位置依次编号，无需把音频秒数换算成 RoPE 坐标。

`configuration_qwen3_5_mllm.py` 中的 `audio_token_id` 记录音频占位标记的编号。文本配置、视觉配置和 `rope_parameters` 继续从原 Qwen3.5 backbone 读取。


```python
from transformers.models.qwen3_5.configuration_qwen3_5 import Qwen3_5Config
from transformers.models.qwen3_omni_moe.configuration_qwen3_omni_moe import (
    Qwen3OmniMoeAudioEncoderConfig,
)


class Qwen3_5MLLMConfig(Qwen3_5Config):
    """在 Qwen3.5 配置中加入音频塔配置。"""

    model_type = "qwen3_5_mllm"
    sub_configs = {
        **Qwen3_5Config.sub_configs,
        "audio_config": Qwen3OmniMoeAudioEncoderConfig,
    }

    def __init__(
        self,
        audio_config = None,
        audio_token_id = None,
        audio_start_token_id = None,
        audio_end_token_id = None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        if audio_config is None:
            audio_config = Qwen3OmniMoeAudioEncoderConfig()
        elif isinstance(audio_config, dict):
            audio_config = Qwen3OmniMoeAudioEncoderConfig.from_dict(
                config_dict = audio_config,
            )
        self.audio_config = audio_config
        self.audio_token_id = audio_token_id
        self.audio_start_token_id = audio_start_token_id
        self.audio_end_token_id = audio_end_token_id
```


### 5.2 音视频排序、文本时间戳与位置编码

联合输入先计算两路的时间，再按时间排列占位标记、加入时间文本，最后调用 Qwen3.5 原生函数计算位置编码。排列在 processor 中完成，模型随后将各塔的特征填入对应位置。

**音视频排序。** 画面和音轨从同一视频文件解码，并减去同一个时间起点。视频使用解码器返回的每帧时间，通过 Qwen3.5 的 `_calculate_timestamps` 取帧组首尾时间的平均值；音频使用卷积网格时间，加上音轨相对这个起点的偏移。两路时间都保留完整精度，交给从 Omni 复用的归并循环：每次取时间较早的一项，时间相等时视频先放。每路内部顺序保持原样，每组视频的空间 token 连在一起。这里复用的是归并算法，比较的数值直接采用上述秒数。[Omni 归并代码](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_omni_moe/processing_qwen3_omni_moe.py)

例如画面与音轨同时开始，实际采样帧时刻为 `0、250、500……ms`（4 fps），每两帧一组。首组视频的平均时间为 `(0 + 250) / 2 = 125 ms`；前两个音频网格位置为 `0、80 ms`。归并结果为 `A0（0 ms）→ A1（80 ms）→ V0（125 ms）→ A2（160 ms）`。

**文本时间戳。** 排序完成后，把同一组秒数按 Qwen3.5 的 `.1f` 格式写成 `<0.1 seconds>`，相邻项显示相同时间时共用一个标签。上例开头为 `<0.0 seconds>[A0]<0.1 seconds>[A1][V0]<0.2 seconds>[A2]`。时间文本随特征进入语言模型，供模型学习内容与时间的关系；舍入仅用于生成文本。[Qwen3.5 使用的视频时间戳处理](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_vl/processing_qwen3_vl.py)

**位置编码。** 完整序列交给 `Qwen3_5Model.get_rope_index`：时间戳文本、起止标记和音频位置使用原生一维编号，视频帧组使用原生网格位置。接入音频时，在模态编号的副本中将音频编号 3 映射到一维分支 0，再调用父类；输入中的音频编号仍保留。视觉位置规则、语言模型的 `rope_parameters` 和生成时的 `rope_deltas` 均沿用 Qwen3.5 原实现。[原生位置计算](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_5/modeling_qwen3_5.py)


### 5.3 构造联合输入序列

这一步涉及三个文件。`media_io.py` 从同一个视频文件读取画面、音轨和时间；`audio_video_utils.py` 计算音频网格时间并归并两路位置；`processing_qwen3_5_mllm.py` 构造带时间戳的联合序列。图像、视频像素处理继续由 Qwen3.5 原组件完成，Mel 提取和输出长度计算沿用官方音频组件。

`use_audio_in_video = True` 时，`videos` 接收带音轨的视频文件路径，每条输入对应一个文件。音频波形、视频帧及其时间信息都在内部从该文件读取；另传的 `audio`、`video_metadata` 和帧数组会被拒绝。没有音轨的视频也会报错。独立音频或独立视频使用普通输入路径。

联合序列放在消息的视频标记处。Processor 按归并结果逐项放入时间文本和占位标记，返回 token 序列、模态编号以及各塔需要的输入特征。

```text
同一个带音轨的视频文件
    ├── 画面 → Qwen3.5 原生抽帧与像素处理 → 视觉塔
    └── 音轨 → 单声道波形 → Mel 特征 → 音频塔 → projector
两路时间 → 交错占位序列 → 填入对应特征 → 语言模型
```


`audio_video_utils.py` 的第一个函数计算音频输出位置对应的时间。当前配置 `n_window = 50`，官方音频塔每 `2 × n_window = 100` 个 Mel 帧分块；Mel 帧间隔为 `160 / 16000 = 0.01` 秒，所以每个满块包含对应 1 秒时长的 100 个 Mel 帧，下一块从第 100 帧开始。最后一块按剩余帧数处理。[官方分块代码](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_omni_moe/modeling_qwen3_omni_moe.py)

每块输出长度调用官方函数。三层步长为 2 的卷积对应每 8 个 Mel 帧一个输出位置，因此满块的网格时间为 `0、80、160……960 ms`，下一块从 `1000 ms` 开始。函数从配置读取分块大小和 Mel 帧间隔，返回用于生成音频时间文本的秒数。

```python
import numpy as np
from transformers.models.qwen3_omni_moe.modeling_qwen3_omni_moe import (
    _get_feat_extract_output_lengths,
)


def get_audio_token_timestamps(
    feature_length,
    n_window,
    hop_length,
    sampling_rate,
):
    """按原音频塔的分块和三层卷积，计算输出位置对应的秒数。"""
    chunk_frames = n_window * 2
    timestamps = []
    for chunk_start in range(0, int(feature_length), chunk_frames):
        chunk_length = min(chunk_frames, int(feature_length) - chunk_start)
        output_length = int(_get_feat_extract_output_lengths(
            input_lengths = chunk_length,
            n_window = n_window,
        ))
        frame_indices = chunk_start + np.arange(stop = output_length) * (2 ** 3)
        timestamps.extend((frame_indices * hop_length / sampling_rate).tolist())
    return timestamps
```


同一文件的 `merge_audio_video_order` 接收两路未舍入的秒数，复用 Omni 的逐项比较循环。时间较早的先放，相等时视频先放，返回每一步该取哪一路的第几个位置。

```python
def merge_audio_video_order(audio_times, video_times):
    """按未舍入的秒数归并两路位置，时间相等时视频先放。"""
    video_index = 0
    audio_index = 0
    order = []
    while video_index < len(video_times) and audio_index < len(audio_times):
        if video_times[video_index] <= audio_times[audio_index]:
            order.append(("video", video_index))
            video_index += 1
        else:
            order.append(("audio", audio_index))
            audio_index += 1
    order.extend(("video", index) for index in range(video_index, len(video_times)))
    order.extend(("audio", index) for index in range(audio_index, len(audio_times)))
    return order
```


`media_io.py` 使用 TorchCodec 解码，需要环境中安装 TorchCodec 和 FFmpeg。两个解码器读取同一路径：视频抽帧调用 Qwen3.5 的 `sample_frames`，音频转为指定采样率的单声道波形。解码器用 `pts_seconds` 返回视频帧和音频样本的播放时间。两路减去同一个起点，保留声音相对画面的偏移；例如音轨晚 0.5 秒开始，首个音频位置就从 0.5 秒参与排列。

`_check_audio_timeline` 检查音轨时间是否连续，遇到时间断点会报错，避免音视频错位。`load_audio_video` 返回的数据用于后面的 processor。

[视频解码接口](https://meta-pytorch.org/torchcodec/stable/generated_examples/decoding/basic_example.html)、[音频解码接口](https://meta-pytorch.org/torchcodec/stable/generated_examples/decoding/audio_decoding.html)。

```python
import json
import subprocess
from fractions import Fraction
from pathlib import Path

import numpy as np
from transformers.video_utils import VideoMetadata



def _check_audio_timeline(video_path, stream_index):
    """检查音轨逐帧时间连续，避免解码器把时间空洞直接拼接掉。"""
    try:
        result = subprocess.run(
            args = [
                "ffprobe", "-v", "error", "-select_streams", str(stream_index),
                "-show_frames", "-show_streams", "-show_entries",
                "frame=pts,nb_samples:stream=time_base,sample_rate",
                "-of", "json", "-i", str(video_path.resolve()),
            ],
            capture_output = True,
            text = True,
            check = True,
        )
    except FileNotFoundError as error:
        raise RuntimeError("读取联合音视频需要 FFmpeg 提供的 ffprobe。") from error
    info = json.loads(s = result.stdout)
    stream = info["streams"][0]
    time_base = Fraction(numerator = stream["time_base"])
    sample_rate = int(stream["sample_rate"])
    tolerance = time_base + Fraction(numerator = 1, denominator = sample_rate)
    first_start = None
    sample_count = 0
    for frame in info["frames"]:
        if "pts" not in frame or "nb_samples" not in frame:
            raise ValueError("音轨缺少时间信息，无法进行音视频联合排列。")
        start = int(frame["pts"]) * time_base
        if first_start is None:
            first_start = start
        expected_start = first_start + Fraction(
            numerator = sample_count, denominator = sample_rate,
        )
        if abs(start - expected_start) > tolerance:
            raise ValueError("当前联合输入要求连续音轨；此文件的音频时间戳存在间断。")
        sample_count += int(frame["nb_samples"])


def load_audio_video(
    video_path,
    video_processor,
    sampling_rate,
    fps = None,
    num_frames = None,
    do_sample_frames = True,
):
    """从同一视频文件读取画面、音轨和时间，保留两路起始时间差。"""
    from torchcodec.decoders import AudioDecoder, VideoDecoder

    video_path = Path(video_path).expanduser()
    if not video_path.is_file():
        raise ValueError(f"找不到视频文件：{video_path}")
    try:
        video_decoder = VideoDecoder(source = video_path, dimension_order = "NHWC")
        audio_decoder = AudioDecoder(
            source = video_path,
            sample_rate = sampling_rate,
            num_channels = 1,
        )
    except (RuntimeError, ValueError) as error:
        raise ValueError(
            f"联合输入需要同一文件内可解码的视频轨道和音频轨道：{video_path}",
        ) from error

    _check_audio_timeline(video_path = video_path, stream_index = audio_decoder.stream_index)

    metadata = VideoMetadata(
        total_num_frames = video_decoder.metadata.num_frames,
        fps = video_decoder.metadata.average_fps,
        width = video_decoder.metadata.width,
        height = video_decoder.metadata.height,
        duration = video_decoder.metadata.duration_seconds,
        video_backend = "torchcodec",
    )
    if do_sample_frames:
        frame_indices = video_processor.sample_frames(
            metadata = metadata,
            fps = fps,
            num_frames = num_frames,
        )
    else:
        frame_indices = np.arange(stop = metadata.total_num_frames)
    frames = video_decoder.get_frames_at(indices = frame_indices.tolist())
    samples = audio_decoder.get_samples_played_in_range(
        start_seconds = audio_decoder.metadata.begin_stream_seconds,
    )
    if frames.data.shape[0] == 0 or samples.data.shape[-1] == 0:
        raise ValueError("联合输入的视频轨道和音频轨道都需要包含有效数据。")

    time_origin = min(float(frames.pts_seconds[0]), samples.pts_seconds)
    metadata.frames_indices = frame_indices.tolist()
    return {
        "video": frames.data.contiguous(),
        "audio": samples.data.squeeze(dim = 0).contiguous().numpy(),
        "metadata": metadata,
        "video_timestamps": (frames.pts_seconds - time_origin).tolist(),
        "audio_start_seconds": samples.pts_seconds - time_origin,
    }
```


下面进入 `processing_qwen3_5_mllm.py`。`_waveform_lengths` 取出每段波形的采样点数，后面用它计算每段音频的有效 Mel 帧数。内部类使用解码后的逐帧时间与音频网格时间归并，再将这些时间格式化为文本。视频时间已经以秒为单位，调用原平均函数时除数取 `1.0`。相邻项显示相同时间时共用一个时间标签，每个视频帧组保留完整的视觉起止标记。

```python
from pathlib import Path

import numpy as np
from transformers import ProcessorMixin, Qwen3VLProcessor
from transformers.feature_extraction_utils import BatchFeature
from transformers.models.qwen3_omni_moe.modeling_qwen3_omni_moe import (
    _get_feat_extract_output_lengths,
)

from .audio_video_utils import get_audio_token_timestamps, merge_audio_video_order
from .media_io import load_audio_video


def _waveform_lengths(audio):
    """返回每段波形的采样点数，批次的写法与 WhisperFeatureExtractor 的判断一致。"""
    if isinstance(audio, np.ndarray) and audio.ndim > 1:
        return [audio.shape[-1]] * audio.shape[0]
    if isinstance(audio, (list, tuple)) and len(audio) > 0 and not np.isscalar(audio[0]):
        return [np.asarray(a = wave).shape[-1] for wave in audio]
    return [np.asarray(a = audio).shape[-1]]


class _Qwen3_5AudioVideoProcessor(Qwen3VLProcessor):
    """只扩展联合音视频的占位序列，复用原有图像和视频预处理。"""

    def replace_video_token(self, video_inputs, video_idx, **kwargs):
        grid = video_inputs["video_grid_thw"][video_idx]
        frame_length = int(grid[1] * grid[2]) // (self.video_processor.merge_size ** 2)

        # 第一步：取得两路未舍入的时间，单位均为秒。
        audio_times = self.audio_timestamps[video_idx]
        # 解码时间已经以秒为单位，除数取 1，复用原函数的帧组平均计算。
        video_times = self._calculate_timestamps(
            indices = list(self.video_timestamps[video_idx]),
            video_fps = 1.0,
            merge_size = self.video_processor.temporal_patch_size,
        )

        # 第二步：复用 Omni 的归并循环，按时间从早到晚排列。
        order = merge_audio_video_order(
            audio_times = audio_times,
            video_times = video_times,
        )

        # 第三步：按 Qwen3.5 格式写入时间文本，相邻同标签共用。
        timestamps = {"video": video_times, "audio": audio_times}
        time_labels = [
            f"<{timestamps[modality][index]:.1f} seconds>"
            for modality, index in order
        ]

        parts = [self.tokenizer.audio_bos_token]
        previous_label = None
        for (modality, index), label in zip(order, time_labels):
            if label != previous_label:
                parts.append(label)
                previous_label = label
            if modality == "video":
                parts.append(
                    self.vision_start_token
                    + self.video_token * frame_length
                    + self.vision_end_token,
                )
            else:
                parts.append(self.tokenizer.audio_token)
        parts.append(self.tokenizer.audio_eos_token)
        return "".join(parts)

```


对外使用的 `Qwen3_5MLLMProcessor` 保存原有的 tokenizer、图像处理器、视频处理器和音频特征提取器；`vision_processor` 用于独立图像、视频和文本。

```python
class Qwen3_5MLLMProcessor(ProcessorMixin):
    """保留 Qwen3.5 的文本、图像和视频处理，并加入音频处理。"""

    def __init__(
        self,
        image_processor,
        tokenizer,
        video_processor,
        feature_extractor,
        chat_template = None,
        audio_n_window = 50,
    ):
        super().__init__(
            image_processor = image_processor,
            tokenizer = tokenizer,
            video_processor = video_processor,
            feature_extractor = feature_extractor,
            chat_template = chat_template,
        )
        self.audio_n_window = audio_n_window
        self.audio_token = tokenizer.audio_token
        self.audio_start_token = tokenizer.audio_bos_token
        self.audio_end_token = tokenizer.audio_eos_token

    @property
    def vision_processor(self):
        return Qwen3VLProcessor(
            image_processor = self.image_processor,
            tokenizer = self.tokenizer,
            video_processor = self.video_processor,
            chat_template = self.chat_template,
        )

```


`__call__` 的联合输入分支先检查文件来源，再调用 `load_audio_video`。画面按原生方法抽帧一次，音轨转成音频特征提取器需要的采样率。随后提取 Mel 特征并展开音频占位标记。纯音频继续接收独立波形。

音频占位要展开成多少个 `<|audio_pad|>`，由每段音频的有效 Mel 帧数决定。Mel 帧间隔为 160 个采样点，单独处理一段音频时，特征提取器给出「采样点数 ÷ 160」向下取整的帧数；几段音频补齐成一批时，它只修正批内最长的一条，其余音频的有效帧按向上取整计算，可能多出 1 帧，于是同一段音频的音频 token 数会随批次组合变化。这里按每段音频自己的采样点数向下取整，重写有效帧的标记 `feature_attention_mask`，单独处理和成批处理得到相同的帧数。

```python
    def __call__(
        self,
        text = None,
        images = None,
        videos = None,
        audio = None,
        sampling_rate = None,
        return_tensors = "pt",
        use_audio_in_video = False,
        **kwargs,
    ):
        if text is None and images is None and videos is None and audio is None:
            raise ValueError("请提供 text、images、videos 或 audio。")
        decoded_media = None
        if use_audio_in_video:
            if audio is not None:
                raise ValueError("联合排列只使用视频文件自带的音轨，请勿另传 audio。")
            if kwargs.get("video_metadata") is not None:
                raise ValueError("联合输入的帧信息从视频文件读取，请勿另传 video_metadata。")
            video_paths = [videos] if isinstance(videos, (str, Path)) else videos
            if not isinstance(video_paths, (list, tuple)) or not video_paths or not all(
                isinstance(path, (str, Path)) for path in video_paths
            ):
                raise ValueError("联合排列需要带音轨的视频文件路径，或每条输入一个文件的路径列表。")
            sampling_rate = self.feature_extractor.sampling_rate
            do_sample_frames = kwargs.pop(
                "do_sample_frames", self.video_processor.do_sample_frames,
            )
            decoded_media = [
                load_audio_video(
                    video_path = path,
                    video_processor = self.video_processor,
                    sampling_rate = sampling_rate,
                    fps = kwargs.get("fps"),
                    num_frames = kwargs.get("num_frames"),
                    do_sample_frames = do_sample_frames,
                )
                for path in video_paths
            ]
            videos = [item["video"] for item in decoded_media]
            audio = [item["audio"] for item in decoded_media]
            kwargs["video_metadata"] = [item["metadata"] for item in decoded_media]
            kwargs["do_sample_frames"] = False
            if text is None:
                text = ["<|vision_start|><|video_pad|><|vision_end|>"] * len(video_paths)

        audio_inputs = None
        if audio is not None:
            if sampling_rate is None:
                raise ValueError("传入 audio 时，请用 sampling_rate 指明波形的采样率。")
            audio_inputs = self.feature_extractor(
                raw_speech = audio,
                sampling_rate = sampling_rate,
                padding = "longest",
                truncation = False,
                return_attention_mask = True,
                return_tensors = return_tensors,
            )
            # WhisperFeatureExtractor 把采样点级的 mask 每隔 hop_length 取一个值，得到「采样点数 ÷ hop_length」
            # 向上取整的帧数，只在批次补齐后的总长度不能整除时去掉最后一列，因此只修正了批内最长的一条。
            # 同一段音频单独处理得到向下取整的帧数，成批处理时可能多 1 帧、多 1 个音频 token。
            # 这里按每段音频自己的采样点数向下取整，重写有效帧的 mask，单独处理与成批处理结果一致。
            hop_length = self.feature_extractor.hop_length
            feature_lens = np.asarray(a = [length // hop_length for length in _waveform_lengths(audio = audio)])
            feature_mask = audio_inputs["attention_mask"]
            feature_mask[:] = 0
            for index, length in enumerate(feature_lens):
                feature_mask[index, :length] = 1
            audio_lengths = _get_feat_extract_output_lengths(
                input_lengths = feature_lens,
                n_window = self.audio_n_window,
            ).tolist()
            if text is None:
                text = [""] * len(audio_lengths)
            elif isinstance(text, str):
                text = [text]
            if len(text) != len(audio_lengths):
                raise ValueError("当前每条文本输入对应一段音频。")
            audio_marker = (
                self.audio_start_token + self.audio_token + self.audio_end_token
            )
            expanded_text = []
            for prompt, audio_length in zip(text, audio_lengths):
                if use_audio_in_video:
                    if prompt.count("<|video_pad|>") != 1 or prompt.count(self.audio_token) > 1:
                        raise ValueError("每条联合输入需要一个视频标记和至多一个音频标记。")
                    prompt = prompt.replace(audio_marker, "")
                    expanded_text.append(prompt.replace(self.audio_token, ""))
                    continue
                if self.audio_token not in prompt:
                    prompt = audio_marker + prompt
                if prompt.count(self.audio_token) != 1:
                    raise ValueError("每条文本请保留一个 <|audio_pad|> 音频标记。")
                expanded_text.append(
                    prompt.replace(self.audio_token, self.audio_token * audio_length),
                )
            text = expanded_text

```


联合输入创建内部 processor，把音轨的起始偏移加到音频网格时间上，并传入解码得到的视频帧时间。两路使用同一个时间起点，随后交给原视觉组件处理像素、原音频组件处理 Mel 特征。

```python
        inputs = BatchFeature(data = {})
        vision_processor = self.vision_processor
        if use_audio_in_video:
            kwargs["return_metadata"] = True
            vision_processor = _Qwen3_5AudioVideoProcessor(
                image_processor = self.image_processor,
                tokenizer = self.tokenizer,
                video_processor = self.video_processor,
                chat_template = self.chat_template,
            )
            vision_processor.audio_timestamps = []
            for length, item in zip(feature_lens, decoded_media):
                audio_times = get_audio_token_timestamps(
                    feature_length = length,
                    n_window = self.audio_n_window,
                    hop_length = self.feature_extractor.hop_length,
                    sampling_rate = self.feature_extractor.sampling_rate,
                )
                audio_times = [
                    timestamp + item["audio_start_seconds"] for timestamp in audio_times
                ]
                vision_processor.audio_timestamps.append(audio_times)
            vision_processor.video_timestamps = [
                item["video_timestamps"] for item in decoded_media
            ]
        if text is not None or images is not None or videos is not None:
            inputs = vision_processor(
                text = text,
                images = images,
                videos = videos,
                return_tensors = return_tensors,
                **kwargs,
            )

```


将音频特征放入返回结果，并标记模态编号：0 为文本、1 为图像、2 为视频、3 为音频。模型按各模态的 token ID 找到特征填入位置。

```python
        if audio_inputs is not None:
            inputs["input_features"] = audio_inputs["input_features"]
            inputs["feature_attention_mask"] = audio_inputs["attention_mask"]
            audio_token_id = self.tokenizer.convert_tokens_to_ids(tokens = self.audio_token)
            token_types = []
            for ids, types, length in zip(
                inputs["input_ids"], inputs["mm_token_type_ids"], audio_lengths,
            ):
                audio_mask = np.asarray(a = ids) == audio_token_id
                if int(audio_mask.sum()) != length:
                    raise ValueError("分词后音频标记不完整，请保留整段音频标记。")
                row_types = np.asarray(a = types).copy()
                row_types[audio_mask] = 3
                token_types.append(row_types.tolist())
            inputs["mm_token_type_ids"] = token_types
```


最后将返回值整理成模型需要的张量。时间已通过排列顺序和时间戳文本保存在输入中，`model_input_names` 列出模型实际接收的字段。

```python
        if use_audio_in_video:
            inputs.pop("video_metadata")

        if audio_inputs is not None:
            inputs = BatchFeature(data = inputs.data, tensor_type = return_tensors)

        return inputs

    @property
    def model_input_names(self):
        return self.vision_processor.model_input_names + [
            "input_features",
            "feature_attention_mask",
        ]
```


### 5.4 定义多模态主干

`modeling_qwen3_5_mllm.py` 中的 `Qwen3_5MLLMModel` 继承官方 `Qwen3_5Model`，保留 `visual` 和 `language_model`，并加入 `audio_tower` 与 `audio_projector`。

两个塔各自按原有顺序输出特征。融合时，沿用 Omni 的 `masked_scatter` 做法：依次把音频向量填进音频占位位置，把视觉向量填进视觉占位位置。归并保留了每一路的内部顺序，因此输出特征无需再次排序。视觉特征的提取和填入直接调用 Qwen3.5 的原有流程。[官方特征填入代码](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3_omni_moe/modeling_qwen3_omni_moe.py)

`get_rope_index` 完成音频模态的接口适配后，直接调用原生 Qwen3.5 方法计算整条联合序列的位置。`_init_weights` 恢复音频塔按公式生成的固定位置表，保证整个模型重新加载时也能使用它。


先导入官方组件，并保留第三节的 SiLU projector。以下代码块按顺序组成同一个 `modeling_qwen3_5_mllm.py`。

```python
import torch
from torch import nn
from transformers.models.qwen3_5.modeling_qwen3_5 import (
    Qwen3_5ForConditionalGeneration,
    Qwen3_5Model,
    Qwen3_5PreTrainedModel,
)
from transformers.models.qwen3_omni_moe.modeling_qwen3_omni_moe import (
    Qwen3OmniMoeAudioEncoder,
    SinusoidsPositionEmbedding,
    _get_feat_extract_output_lengths,
)

from .configuration_qwen3_5_mllm import Qwen3_5MLLMConfig


class Qwen3_5MLLMAudioProjector(nn.Module):
    """将音频塔输出映射到语言模型的特征维度。"""

    def __init__(self, audio_output_dim, text_hidden_size):
        super().__init__()
        self.linear_1 = nn.Linear(
            in_features = audio_output_dim,
            out_features = text_hidden_size,
            bias = True,
        )
        self.silu = nn.SiLU()
        self.linear_2 = nn.Linear(
            in_features = text_hidden_size,
            out_features = text_hidden_size,
            bias = True,
        )

    def forward(self, audio_features):
        hidden_states = self.linear_1(input = audio_features)
        hidden_states = self.silu(input = hidden_states)
        return self.linear_2(input = hidden_states)
```


父类建立视觉塔和语言模型，随后加入音频塔和 projector。`_init_weights` 恢复固定位置表；后续方法继续写在这个类内。

```python
class Qwen3_5MLLMModel(Qwen3_5Model):
    """在 Qwen3.5 的视觉和语言主干中加入音频塔与 projector。"""

    config_class = Qwen3_5MLLMConfig
    base_model_prefix = "model"
    input_modalities = ("text", "image", "video", "audio")
    _can_compile_fullgraph = False
    _no_split_modules = [
        "Qwen3_5DecoderLayer",
        "Qwen3_5VisionBlock",
        "Qwen3OmniMoeAudioEncoderLayer",
    ]

    def __init__(self, config):
        super().__init__(config = config)
        self.audio_tower = Qwen3OmniMoeAudioEncoder(config = config.audio_config)
        self.audio_projector = Qwen3_5MLLMAudioProjector(
            audio_output_dim = config.audio_config.output_dim,
            text_hidden_size = config.text_config.hidden_size,
        )
        self.post_init()

    @torch.no_grad()
    def _init_weights(self, module):
        super()._init_weights(module = module)
        if isinstance(module, SinusoidsPositionEmbedding):
            positions = module.compute_default_singular_positional_embedding()
            module.positional_embedding.copy_(other = positions)
```


`get_rope_index` 只做音频模态的接口适配：在模态编号的副本里，将音频编号 3 映射到原生的一维位置分支 0，然后调用父类。所有位置数值和 `rope_deltas` 都由 Qwen3.5 原函数计算；输入中的音频模态编号仍然保留。

```python
    def get_rope_index(
        self,
        input_ids,
        mm_token_type_ids = None,
        image_grid_thw = None,
        video_grid_thw = None,
        attention_mask = None,
        **kwargs,
    ):
        if mm_token_type_ids is None:
            mm_token_type_ids = torch.zeros_like(input = input_ids)
            mm_token_type_ids[input_ids == self.config.image_token_id] = 1
            mm_token_type_ids[input_ids == self.config.video_token_id] = 2
            if self.config.audio_token_id is not None:
                mm_token_type_ids[input_ids == self.config.audio_token_id] = 3
        # 音频沿用原生的一维位置规则；保留原模态编号供特征填入使用。
        position_types = mm_token_type_ids.clone()
        position_types[position_types == 3] = 0
        return super().get_rope_index(
            input_ids = input_ids,
            mm_token_type_ids = position_types,
            image_grid_thw = image_grid_thw,
            video_grid_thw = video_grid_thw,
            attention_mask = attention_mask,
        )
```


`get_audio_features` 取出有效 Mel 帧，调用原音频塔，再经过 projector，得到语言模型需要的 1024 维向量。

```python
    def get_audio_features(self, input_features, feature_attention_mask):
        feature_mask = feature_attention_mask.to(
            device = input_features.device,
            dtype = torch.bool,
        )
        feature_lens = feature_mask.sum(dim = -1)
        mel_features = input_features.transpose(dim0 = 1, dim1 = 2)
        packed_features = mel_features[feature_mask]
        packed_features = packed_features.transpose(dim0 = 0, dim1 = 1).contiguous()
        packed_features = packed_features.to(
            device = self.audio_tower.device,
            dtype = self.audio_tower.dtype,
        )
        audio_outputs = self.audio_tower(
            input_features = packed_features,
            feature_lens = feature_lens.to(device = self.audio_tower.device),
        )
        projector_weight = self.audio_projector.linear_1.weight
        audio_features = audio_outputs.last_hidden_state.to(
            device = projector_weight.device,
            dtype = projector_weight.dtype,
        )
        return self.audio_projector(audio_features = audio_features)
```


`forward` 按音频 token 编号找到占位位置，用 `masked_scatter` 依次填入音频向量，再交给父类处理视觉特征和语言模型。位置计算与特征填入使用同一条输入序列。

```python
    def forward(
        self,
        input_ids = None,
        attention_mask = None,
        position_ids = None,
        past_key_values = None,
        inputs_embeds = None,
        pixel_values = None,
        pixel_values_videos = None,
        image_grid_thw = None,
        video_grid_thw = None,
        mm_token_type_ids = None,
        input_features = None,
        feature_attention_mask = None,
        **kwargs,
    ):
        if input_features is not None:
            if input_ids is None or inputs_embeds is not None:
                raise ValueError("传入音频时，请用 input_ids 指明音频标记的位置。")
            if feature_attention_mask is None or self.config.audio_token_id is None:
                raise ValueError("音频输入需要 feature_attention_mask 和 audio_token_id。")
            audio_mask = input_ids == self.config.audio_token_id
            feature_lens = feature_attention_mask.sum(dim = -1)
            audio_lengths = _get_feat_extract_output_lengths(
                input_lengths = feature_lens,
                n_window = self.config.audio_config.n_window,
            )
            actual_lengths = audio_mask.sum(dim = -1)
            if actual_lengths.shape != audio_lengths.shape or not torch.equal(
                input = actual_lengths,
                other = audio_lengths.to(device = actual_lengths.device),
            ):
                raise ValueError("每条输入的音频标记数量必须与音频特征数量一致。")
            inputs_embeds = self.get_input_embeddings()(input = input_ids)
            audio_features = self.get_audio_features(
                input_features = input_features,
                feature_attention_mask = feature_attention_mask,
            )
            audio_features = audio_features.to(
                device = inputs_embeds.device,
                dtype = inputs_embeds.dtype,
            )
            inputs_embeds = inputs_embeds.masked_scatter(
                mask = audio_mask.unsqueeze(dim = -1),
                source = audio_features,
            )
            if position_ids is None:
                position_ids, self.rope_deltas = self.get_rope_index(
                    input_ids = input_ids,
                    mm_token_type_ids = mm_token_type_ids,
                    image_grid_thw = image_grid_thw,
                    video_grid_thw = video_grid_thw,
                    attention_mask = attention_mask,
                )
            input_ids = None

        return super().forward(
            input_ids = input_ids,
            attention_mask = attention_mask,
            position_ids = position_ids,
            past_key_values = past_key_values,
            inputs_embeds = inputs_embeds,
            pixel_values = pixel_values,
            pixel_values_videos = pixel_values_videos,
            image_grid_thw = image_grid_thw,
            video_grid_thw = video_grid_thw,
            mm_token_type_ids = mm_token_type_ids,
            **kwargs,
        )
```


### 5.5 接上文本输出层

同一文件中的 `Qwen3_5MLLMForConditionalGeneration` 将主干放到 `self.model`，并保留 `lm_head`。音频、视觉与语言参数的名称分别是 `model.audio_tower.*`、`model.visual.*`、`model.language_model.*`，projector 使用 `model.audio_projector.*`。

外层 `forward` 把音频特征、视觉输入和 token 序列传给主干，文本分数和训练损失沿用 Qwen3.5 的计算。`generate` 沿用原实现：首次处理完整输入，后续利用缓存和已有位置偏移生成文本。下面这段接在同一文件的主干类之后。




```python
class Qwen3_5MLLMForConditionalGeneration(Qwen3_5ForConditionalGeneration):
    """使用融合后的多模态主干，沿用 Qwen3.5 的文本输出与生成方法。"""

    config_class = Qwen3_5MLLMConfig
    base_model_prefix = "model"
    input_modalities = ("text", "image", "video", "audio")
    _can_compile_fullgraph = False
    _no_split_modules = Qwen3_5MLLMModel._no_split_modules

    def __init__(self, config):
        Qwen3_5PreTrainedModel.__init__(self = self, config = config)
        self.model = Qwen3_5MLLMModel(config = config)
        self.lm_head = nn.Linear(
            in_features = config.text_config.hidden_size,
            out_features = config.text_config.vocab_size,
            bias = False,
        )
        self.post_init()

    @torch.no_grad()
    def _init_weights(self, module):
        super()._init_weights(module = module)
        if isinstance(module, SinusoidsPositionEmbedding):
            positions = module.compute_default_singular_positional_embedding()
            module.positional_embedding.copy_(other = positions)

    def forward(
        self,
        input_ids = None,
        attention_mask = None,
        position_ids = None,
        past_key_values = None,
        inputs_embeds = None,
        labels = None,
        pixel_values = None,
        pixel_values_videos = None,
        image_grid_thw = None,
        video_grid_thw = None,
        mm_token_type_ids = None,
        input_features = None,
        feature_attention_mask = None,
        logits_to_keep = 0,
        **kwargs,
    ):
        return super().forward(
            input_ids = input_ids,
            attention_mask = attention_mask,
            position_ids = position_ids,
            past_key_values = past_key_values,
            inputs_embeds = inputs_embeds,
            labels = labels,
            pixel_values = pixel_values,
            pixel_values_videos = pixel_values_videos,
            image_grid_thw = image_grid_thw,
            video_grid_thw = video_grid_thw,
            mm_token_type_ids = mm_token_type_ids,
            input_features = input_features,
            feature_attention_mask = feature_attention_mask,
            logits_to_keep = logits_to_keep,
            **kwargs,
        )
```


### 5.6 用对象更新配置与对话模板

从 tokenizer 读取音频标记编号，写入配置对象。位置编码继续使用原 Qwen3.5 的配置。

对话模板增加一个 `audio` 分支，将消息中的音频项转换为音频标记。原有文本、图像和视频分支继续保留。下面重新创建扩展后的 processor，沿用 Qwen3.5 的原有输入处理组件。


In [20]:
from transformers import Qwen3_5ForConditionalGeneration, AutoModelForImageTextToText

mllm_config.audio_token_id = mllm_processor.tokenizer.convert_tokens_to_ids(
    tokens = "<|audio_pad|>",
)

audio_template_branch = """{%- elif item.type == 'audio' %}
                {{- '<|audio_start|><|audio_pad|><|audio_end|>' }}
            """
text_template_branch = "{%- elif 'text' in item %}"
mllm_chat_template = backbone_processor.chat_template.replace(
    text_template_branch,
    audio_template_branch + text_template_branch,
)
mllm_processor = Qwen3_5MLLMProcessor(
    image_processor = backbone_processor.image_processor,
    tokenizer = backbone_processor.tokenizer,
    video_processor = backbone_processor.video_processor,
    feature_extractor = audio_processor,
    chat_template = mllm_chat_template,
    audio_n_window = mllm_config.audio_config.n_window,
)

print("音频标记编号:", mllm_config.audio_token_id)


音频标记编号: 248076


### 5.7 装入各组件的参数

加载带 `lm_head` 的原始 Qwen3.5 模型，再创建完整 MLLM。下面直接将已有模块赋给对应属性：视觉塔、语言主干、文本输出层来自 Qwen3.5，音频塔来自第一节导出的模型，projector 沿用第四节的对象。

Qwen3.5-0.8B 的输入词向量和文本输出层共享权重，`tie_weights()` 保持这个关系。示例继续使用 CPU 上的 FP32。Projector 仍需后续训练，才能学会让语言模型理解音频特征。


In [21]:
Qwen3_5MLLMForConditionalGeneration = get_class_from_dynamic_module(
    class_reference = "modeling_qwen3_5_mllm.Qwen3_5MLLMForConditionalGeneration",
    pretrained_model_name_or_path = mllm_dir,
)
backbone_with_head = Qwen3_5ForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path = "weights/Qwen3.5-0.8B",
    dtype = torch.float32,
    attn_implementation = "eager",
)
full_model = Qwen3_5MLLMForConditionalGeneration(config = mllm_config)
full_model.model.visual = backbone_with_head.model.visual
full_model.model.language_model = backbone_with_head.model.language_model
full_model.lm_head = backbone_with_head.lm_head
full_model.model.audio_tower = audio_component.model.audio_tower
full_model.model.audio_projector = mllm_projector
full_model.tie_weights()
full_model.eval()

print("视觉塔:", type(full_model.model.visual).__name__)
print("语言模型:", type(full_model.model.language_model).__name__)
print("音频塔:", type(full_model.model.audio_tower).__name__)
print("Projector:", type(full_model.model.audio_projector).__name__)


视觉塔: Qwen3_5VisionModel
语言模型: Qwen3_5TextModel
音频塔: Qwen3OmniMoeAudioEncoder
Projector: Qwen3_5MLLMAudioProjector


### 5.8 准备同一文件的联合音视频输入

`examples/audio_video_demo.mov` 是本项目附带的一秒示例视频，包含 24 帧画面和一秒声音。下面只传入这个文件，由 processor 提取两路数据。`fps` 控制抽帧数量，排序使用所选帧在文件中的播放时间。视觉塔每两帧一组，共 12 个帧组；音轨提取出 100 个有效 Mel 帧，对应 13 个音频位置。

通过消息模板安排视频输入的位置，再交给 processor 展开联合序列。模型调用原生 Qwen3.5 方法生成位置编号。三轴位置张量形状为 `[3, 批量大小, 输入位置数]`，第一维依次对应时间、高度、宽度。


In [2]:
paired_video_path = Path("examples/audio_video_demo.mov")
messages = [{
    "role": "user",
    "content": [
        {"type": "video", "video": str(paired_video_path)},
        {"type": "text", "text": "请描述这段视频及其中的声音。"},
    ],
}]
prompt = mllm_processor.apply_chat_template(
    conversation = messages,
    tokenize = False,
    add_generation_prompt = True,
)
fusion_inputs = mllm_processor(
    text = prompt,
    videos = paired_video_path,
    fps = 24,
    use_audio_in_video = True,
    return_tensors = "pt",
)
position_ids, rope_deltas = full_model.model.get_rope_index(
    input_ids = fusion_inputs["input_ids"],
    mm_token_type_ids = fusion_inputs["mm_token_type_ids"],
    video_grid_thw = fusion_inputs["video_grid_thw"],
    attention_mask = fusion_inputs["attention_mask"],
)
audio_positions_mask = fusion_inputs["mm_token_type_ids"][0] == 3

print("音频位置数:", int(audio_positions_mask.sum()))
print("三轴 position_ids 形状:", position_ids.shape)


音频位置数: 13
三轴 position_ids 形状: torch.Size([3, 1, 175])


下面查看 processor 实际生成的联合序列。为便于阅读，将每组视觉占位标记缩写为 `[video_tokens_i]`，将音频占位标记标成 `[audio_token_i]`；时间文本和交错顺序保留。


In [4]:
joint_sequence = mllm_processor.tokenizer.decode(
    token_ids = fusion_inputs["input_ids"][0],
    skip_special_tokens = False,
)
joint_sequence = joint_sequence.split("<|audio_start|>", 1)[1]
joint_sequence = joint_sequence.split("<|audio_end|>", 1)[0]
frame_tokens = int(fusion_inputs["video_grid_thw"][0, 1:].prod())
frame_tokens = frame_tokens // (mllm_processor.video_processor.merge_size ** 2)
for index in range(int(fusion_inputs["video_grid_thw"][0, 0])):
    joint_sequence = joint_sequence.replace(
        "<|video_pad|>" * frame_tokens,
        f"[video_tokens_{index}]",
        1,
    )
for index in range(int(audio_positions_mask.sum())):
    joint_sequence = joint_sequence.replace("<|audio_pad|>", f"[audio_token_{index}]", 1)
joint_sequence = joint_sequence.replace("<|vision_start|>", "")
joint_sequence = joint_sequence.replace("<|vision_end|>", "")
print(joint_sequence.replace("<", "\n<").strip())


<0.0 seconds>[audio_token_0][video_tokens_0]
<0.1 seconds>[audio_token_1][video_tokens_1]
<0.2 seconds>[audio_token_2][video_tokens_2][audio_token_3]
<0.3 seconds>[video_tokens_3][audio_token_4]
<0.4 seconds>[video_tokens_4][audio_token_5][video_tokens_5]
<0.5 seconds>[audio_token_6][video_tokens_6]
<0.6 seconds>[audio_token_7][video_tokens_7][audio_token_8]
<0.7 seconds>[video_tokens_8][audio_token_9]
<0.8 seconds>[video_tokens_9][audio_token_10]
<0.9 seconds>[video_tokens_10][audio_token_11][video_tokens_11]
<1.0 seconds>[audio_token_12]


### 5.9 调用完整 forward

把 processor 的输出传给完整模型，就会执行音频编码、音频投影、视觉编码和语言模型计算。融合层按已经交错排列的占位位置填入各路特征。`logits` 是词表中各个 token 的分数；`logits_to_keep = 1` 让这个示例只返回最后一个输入位置的分数，形状为 `[批量大小, 1, 词表大小]`。


In [5]:
with torch.inference_mode():
    fusion_outputs = full_model(
        **fusion_inputs,
        use_cache = False,
        logits_to_keep = 1,
    )

print("文本分数形状:", fusion_outputs.logits.shape)


文本分数形状: torch.Size([1, 1, 248320])


### 5.10 保存完整模型并重新加载

登记 `AutoModelForImageTextToText`，让它加载带文本输出层的完整模型；在配置对象的 `auto_map` 中补上基础模型 `AutoModel` 的入口。随后由 `save_pretrained` 保存权重、配置、自定义源码和 processor，相对导入的 `media_io.py` 也会一起保存。`architectures` 会自动记录完整模型类名。

保存后的目录会增加 `model*.safetensors` 权重文件，以及 `generation_config.json`。下面用 `AutoModelForImageTextToText` 重新加载整个模型，加载路径就是这个目录。


In [25]:
Qwen3_5MLLMForConditionalGeneration.register_for_auto_class(
    auto_class = "AutoModelForImageTextToText",
)
full_model.config.auto_map["AutoModel"] = (
    "modeling_qwen3_5_mllm.Qwen3_5MLLMModel"
)
full_model.save_pretrained(save_directory = mllm_dir)
mllm_processor.save_pretrained(save_directory = mllm_dir)

loaded_mllm = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path = mllm_dir,
    trust_remote_code = True,
    dtype = torch.float32,
    attn_implementation = "eager",
)
loaded_processor = AutoProcessor.from_pretrained(
    pretrained_model_name_or_path = mllm_dir,
    trust_remote_code = True,
)
loaded_mllm.eval()

print("加载的完整模型:", type(loaded_mllm).__name__)
print("加载的 processor:", type(loaded_processor).__name__)


加载的完整模型: Qwen3_5MLLMForConditionalGeneration
加载的 processor: Qwen3_5MLLMProcessor


### 5.11 调用 generate

最后用重新加载的模型生成 4 个新 token。`use_cache = True` 保存已有上下文的计算结果，后续生成沿用多模态位置偏移。这里的示例视频用于演示调用，音频 projector 尚未训练，生成内容仅用于观察流程。


In [6]:
with torch.inference_mode():
    generated_ids = loaded_mllm.generate(
        **fusion_inputs,
        max_new_tokens = 4,
        do_sample = False,
        use_cache = True,
        pad_token_id = loaded_processor.tokenizer.pad_token_id,
    )

new_token_ids = generated_ids[:, fusion_inputs["input_ids"].shape[1]:]
generated_text = loaded_processor.batch_decode(
    sequences = new_token_ids,
    skip_special_tokens = True,
)
print("新生成的 token 数:", new_token_ids.shape[1])
print("生成文本:", generated_text)


新生成的 token 数: 4
生成文本: ['这段视频展示了一个']


vLLM 的命名与注册要求见 4.10 节。

参考：[Qwen3.5 模型接口](https://huggingface.co/docs/transformers/model_doc/qwen3_5)、[Hugging Face 自定义模型保存](https://huggingface.co/docs/transformers/custom_models)。


## 6. 登记音频起止特殊 token

`<|audio_start|>` 和 `<|audio_end|>` 标记音频输入的边界。当前 Qwen3.5 tokenizer 已将它们注册为特殊 token，编号分别为 `248070` 和 `248071`。本节通过 Transformers 接口登记这两个标记，将编号写入模型配置，并让 processor 从 tokenizer 读取它们。

起止标记由语言模型的词向量层处理；中间的 `<|audio_pad|>` 位置由音频塔和 projector 的输出填入。这两个已有编号继续使用原词向量，词表大小保持原样。

### 6.1 登记并读取编号

Tokenizer 用 `audio_bos_token` 表示音频开始，用 `audio_eos_token` 表示音频结束。下面的 `add_special_tokens` 会复用已有 token，因此新增数量为 0。接着通过 tokenizer 读取编号，避免把编号直接写进代码。


In [2]:
audio_tokenizer = loaded_processor.tokenizer
added_audio_tokens = audio_tokenizer.add_special_tokens(
    special_tokens_dict = {
        "audio_bos_token": "<|audio_start|>",
        "audio_eos_token": "<|audio_end|>",
    },
)
audio_start_token_id = audio_tokenizer.audio_bos_token_id
audio_end_token_id = audio_tokenizer.audio_eos_token_id

print("新增 token 数:", added_audio_tokens)
print("音频开始:", audio_tokenizer.audio_bos_token, audio_start_token_id)
print("音频结束:", audio_tokenizer.audio_eos_token, audio_end_token_id)


新增 token 数: 0
音频开始: <|audio_start|> 248070
音频结束: <|audio_end|> 248071


### 6.2 在模型配置中保存编号

`configuration_qwen3_5_mllm.py` 的配置类增加 `audio_start_token_id` 和 `audio_end_token_id`。这两个字段记录起止标记的编号，保存后会随配置一起加载。构造方法中的相关字段如下。

```python
    def __init__(
        self,
        audio_config = None,
        audio_token_id = None,
        audio_start_token_id = None,
        audio_end_token_id = None,
        **kwargs,
    ):
        super().__init__(**kwargs)
        if audio_config is None:
            audio_config = Qwen3OmniMoeAudioEncoderConfig()
        elif isinstance(audio_config, dict):
            audio_config = Qwen3OmniMoeAudioEncoderConfig.from_dict(
                config_dict = audio_config,
            )
        self.audio_config = audio_config
        self.audio_token_id = audio_token_id
        self.audio_start_token_id = audio_start_token_id
        self.audio_end_token_id = audio_end_token_id
```


### 6.3 让 processor 使用 tokenizer 中的标记

`processing_qwen3_5_mllm.py` 中，`Qwen3_5MLLMProcessor.__init__` 从 tokenizer 读取三个音频标记。随后 `__call__` 用它们拼出音频占位片段，并按音频特征数量展开中间的占位位置。

```python
self.audio_token = tokenizer.audio_token
self.audio_start_token = tokenizer.audio_bos_token
self.audio_end_token = tokenizer.audio_eos_token

audio_marker = (
    self.audio_start_token + self.audio_token + self.audio_end_token
)
```

同一文件中的 `replace_video_token` 使用这些标记构造联合音视频序列，相关代码如下。

```python
parts = [self.tokenizer.audio_bos_token]
# 归并循环遇到音频位置时：
parts.append(self.tokenizer.audio_token)
# 联合序列结束时：
parts.append(self.tokenizer.audio_eos_token)
```


### 6.4 更新对象并保存

把 tokenizer 读出的编号写入模型配置，再保存 config 和 processor。词向量继续使用已有参数，本步保存的是配置、tokenizer 和自定义源码。


In [3]:
loaded_mllm.config.audio_start_token_id = audio_start_token_id
loaded_mllm.config.audio_end_token_id = audio_end_token_id
loaded_mllm.config.save_pretrained(save_directory = mllm_dir)
loaded_processor.save_pretrained(save_directory = mllm_dir)


['qwen3.5-0.8b-mllm/processor_config.json']

### 6.5 重新加载并查看标记

重新加载保存的 config 和 processor。把一个音频占位片段交给 tokenizer，可以看到开始、占位、结束分别对应一个编号；实际音频输入中的占位数量由 processor 根据音频长度展开。


In [4]:
from transformers import AutoConfig

saved_audio_config = AutoConfig.from_pretrained(
    pretrained_model_name_or_path = mllm_dir,
    trust_remote_code = True,
)
saved_audio_processor = AutoProcessor.from_pretrained(
    pretrained_model_name_or_path = mllm_dir,
    trust_remote_code = True,
)
audio_marker = (
    saved_audio_processor.audio_start_token
    + saved_audio_processor.audio_token
    + saved_audio_processor.audio_end_token
)
audio_marker_ids = saved_audio_processor.tokenizer.encode(
    text = audio_marker,
    add_special_tokens = False,
)

print("配置中的起止编号:", saved_audio_config.audio_start_token_id, saved_audio_config.audio_end_token_id)
print("音频占位片段:", audio_marker)
print("对应 token 编号:", audio_marker_ids)


配置中的起止编号: 248070 248071
音频占位片段: <|audio_start|><|audio_pad|><|audio_end|>
对应 token 编号: [248070, 248076, 248071]
